# k-NN Ensemble with Trained DINOv2 Classifier

**Strategy:** Combine the trained DINOv2 classifier (unfreeze-1) with a k-NN classifier on DINOv2 features, then average their predictions. k-NN and the linear head make different kinds of errors — k-NN uses local feature similarity while the head learns global decision boundaries.

**Baseline to beat:** DINOv2 unfreeze-1 + threshold → Macro F1: 0.8992, Accuracy: 0.9000

---

## 1. Setup

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_recall_fscore_support
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import normalize

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Data

In [ ]:
TRAIN_DIR = './data_train'
TEST_DIR = './data_test'

CLASS_NAMES = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
NUM_CLASSES = len(CLASS_NAMES)
CLEAN_CLASS_NAMES = [c for c in CLASS_NAMES if c != 'other']
NUM_CLEAN = len(CLEAN_CLASS_NAMES)

print(f'All classes ({NUM_CLASSES}): {CLASS_NAMES}')
print(f'Clean classes ({NUM_CLEAN}): {CLEAN_CLASS_NAMES}')

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 16

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Deterministic transform for feature extraction (no random augmentation)
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [ ]:
# For training the classifier: 5-class (exclude other)
full_train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transform)

other_class_idx = full_train_dataset.class_to_idx['other']
clean_indices = [i for i, (_, label) in enumerate(full_train_dataset.samples) if label != other_class_idx]
clean_subset = Subset(full_train_dataset, clean_indices)

old_to_new = {}
new_idx = 0
for old_idx, cls_name in enumerate(CLASS_NAMES):
    if cls_name != 'other':
        old_to_new[old_idx] = new_idx
        new_idx += 1

# Reverse mapping: 5-class index -> 6-class index
new_to_old = {v: k for k, v in old_to_new.items()}

class RemappedDataset(torch.utils.data.Dataset):
    def __init__(self, subset, label_map):
        self.subset = subset
        self.label_map = label_map
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        image, label = self.subset[idx]
        return image, self.label_map[label]

clean_train_dataset = RemappedDataset(clean_subset, old_to_new)
clean_train_loader = DataLoader(clean_train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                                num_workers=0, pin_memory=True)

# For feature extraction: deterministic transforms, full 6-class
train_eval_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=eval_transform)
test_eval_dataset  = datasets.ImageFolder(root=TEST_DIR,  transform=eval_transform)

# 5-class train set with eval transform (for k-NN fitting)
clean_eval_indices = [i for i, (_, label) in enumerate(train_eval_dataset.samples) if label != other_class_idx]
clean_eval_subset = Subset(train_eval_dataset, clean_eval_indices)
clean_eval_remapped = RemappedDataset(clean_eval_subset, old_to_new)

clean_eval_loader = DataLoader(clean_eval_remapped, batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=0, pin_memory=True)
full_test_loader = DataLoader(test_eval_dataset, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)

print(f'5-class train (augmented): {len(clean_train_dataset)} images')
print(f'5-class train (eval):      {len(clean_eval_remapped)} images')
print(f'6-class test:              {len(test_eval_dataset)} images')

## 3. Model: DINOv2 Unfreeze-1 (Best from Previous Experiment)

In [ ]:
dinov2_backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
print(f'DINOv2 ViT-S loaded. Feature dim: {dinov2_backbone.embed_dim}')

In [ ]:
class DINOv2PartialFinetune(nn.Module):
    def __init__(self, backbone, num_classes, unfreeze_last_n=1, embed_dim=384):
        super().__init__()
        self.backbone = backbone
        num_blocks = len(backbone.blocks)
        
        for param in self.backbone.parameters():
            param.requires_grad = False
        for i in range(num_blocks - unfreeze_last_n, num_blocks):
            for param in self.backbone.blocks[i].parameters():
                param.requires_grad = True
        for param in self.backbone.norm.parameters():
            param.requires_grad = True
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(p=0.3),
            nn.Linear(embed_dim, 128),
            nn.GELU(),
            nn.Dropout(p=0.1),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)
    
    def extract_features(self, x):
        """Return backbone features (before classifier head)."""
        return self.backbone(x)

## 4. Train the Classifier (Unfreeze-1)

Re-train the best configuration from the previous experiment.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0; correct = 0; total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward(); optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return running_loss / total, correct / total

In [ ]:
model = DINOv2PartialFinetune(dinov2_backbone, NUM_CLEAN, unfreeze_last_n=1).to(device)

criterion = nn.CrossEntropyLoss()

backbone_params = list(model.backbone.blocks[11].parameters()) + list(model.backbone.norm.parameters())
optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': 1e-5},
    {'params': model.classifier.parameters(), 'lr': 1e-3},
], weight_decay=1e-4)

scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=1, eta_min=1e-6)

# Train
best_acc = 0.0
PATIENCE = 7
patience_counter = 0

print('Training DINOv2 unfreeze-1...\n')
print(f'{"Epoch":>6} | {"Loss":>8} | {"Acc":>8}')
print('-' * 30)

for epoch in range(1, 31):
    loss, acc = train_one_epoch(model, clean_train_loader, criterion, optimizer, device)
    scheduler.step()
    
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), 'best_model_unfreeze1.pth')
        patience_counter = 0
    else:
        patience_counter += 1
    
    if epoch % 5 == 0 or epoch == 1:
        print(f'{epoch:>6} | {loss:>8.4f} | {acc:>8.4f}')
    
    if patience_counter >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'\nBest train acc: {best_acc:.4f}')

# Load best
model.load_state_dict(torch.load('best_model_unfreeze1.pth', map_location=device))
model.eval()
print('Model loaded.')

## 5. Extract Features for k-NN

Use the fine-tuned backbone to extract features for both train and test sets. The fine-tuned features should be better than raw DINOv2 features since the last block has been adapted to our domain.

In [ ]:
@torch.no_grad()
def extract_features(model, loader, device):
    """Extract backbone features from the fine-tuned model."""
    model.eval()
    all_features, all_labels = [], []
    for images, labels in loader:
        features = model.extract_features(images.to(device))
        all_features.append(features.cpu().numpy())
        all_labels.extend(labels.numpy())
    return np.vstack(all_features), np.array(all_labels)

print('Extracting features with fine-tuned backbone...')

# 5-class train features (for k-NN fitting)
train_features, train_labels_5cls = extract_features(model, clean_eval_loader, device)

# 6-class test features (for evaluation)
test_features, test_labels_6cls = extract_features(model, full_test_loader, device)

# L2 normalize features (important for cosine-based k-NN)
train_features_norm = normalize(train_features, norm='l2')
test_features_norm = normalize(test_features, norm='l2')

print(f'Train features: {train_features_norm.shape} (5-class labels)')
print(f'Test features:  {test_features_norm.shape} (6-class labels)')

## 6. Build k-NN Classifier

Fit a k-NN on the 5-class training features. At inference, k-NN outputs probability estimates (based on neighbor vote proportions) which we can ensemble with the trained classifier's softmax outputs.

In [ ]:
# Find optimal k via leave-one-out style evaluation on training set
from sklearn.model_selection import cross_val_score

k_values = [3, 5, 7, 9, 11, 13, 15]
cv_scores = []

print('Finding optimal k...\n')
print(f'{"k":>4} | {"CV Accuracy (mean +/- std)":>30}')
print('-' * 40)

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, metric='cosine', weights='distance')
    scores = cross_val_score(knn, train_features_norm, train_labels_5cls, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())
    print(f'{k:>4} | {scores.mean():.4f} +/- {scores.std():.4f}')

best_k = k_values[np.argmax(cv_scores)]
print(f'\nBest k: {best_k} (CV accuracy: {max(cv_scores):.4f})')

In [ ]:
# Fit final k-NN with best k
knn_model = KNeighborsClassifier(
    n_neighbors=best_k,
    metric='cosine',       # cosine similarity — works well with normalized features
    weights='distance',    # closer neighbors have more influence
)
knn_model.fit(train_features_norm, train_labels_5cls)

# k-NN predictions on test set (5-class probabilities)
knn_probs_5cls = knn_model.predict_proba(test_features_norm)
print(f'k-NN fitted with k={best_k}')
print(f'k-NN output shape: {knn_probs_5cls.shape} (samples x 5 classes)')

## 7. Get Trained Classifier Probabilities

In [ ]:
@torch.no_grad()
def get_classifier_probs(model, loader, device):
    """Get softmax probabilities from the trained classifier."""
    model.eval()
    all_probs, all_labels = [], []
    for images, labels in loader:
        logits = model(images.to(device))
        probs = torch.softmax(logits, dim=1)
        all_probs.append(probs.cpu().numpy())
        all_labels.extend(labels.numpy())
    return np.vstack(all_probs), np.array(all_labels)

clf_probs_5cls, test_labels_check = get_classifier_probs(model, full_test_loader, device)
print(f'Classifier output shape: {clf_probs_5cls.shape}')

# Verify labels match
assert np.array_equal(test_labels_6cls, test_labels_check), 'Label mismatch!'
print('Labels verified — both sources match.')

## 8. Ensemble: Classifier + k-NN

### Ensemble strategy:
1. Average the 5-class probability distributions from the classifier and k-NN
2. Apply confidence thresholding on the ensembled probabilities
3. If max ensemble confidence < threshold → predict `other`

We also experiment with different ensemble weights (how much to trust each model).

In [ ]:
def ensemble_predict_with_threshold(clf_probs, knn_probs, clf_weight, threshold,
                                     clean_class_names, all_class_names):
    """
    Ensemble classifier + k-NN probabilities, then apply threshold for 'other'.
    
    clf_weight: weight for classifier (k-NN gets 1 - clf_weight)
    """
    other_idx = all_class_names.index('other')
    
    # Weighted average of probabilities
    knn_weight = 1.0 - clf_weight
    ensemble_probs = clf_weight * clf_probs + knn_weight * knn_probs
    
    max_probs = ensemble_probs.max(axis=1)
    pred_5cls = ensemble_probs.argmax(axis=1)
    
    # Map back to 6-class and apply threshold
    new_to_old = {}
    ci = 0
    for oi, cls in enumerate(all_class_names):
        if cls != 'other':
            new_to_old[ci] = oi
            ci += 1
    
    preds_6cls = []
    for i in range(len(max_probs)):
        if max_probs[i] < threshold:
            preds_6cls.append(other_idx)
        else:
            preds_6cls.append(new_to_old[pred_5cls[i]])
    
    return np.array(preds_6cls), max_probs

In [ ]:
# Grid search over ensemble weights and thresholds
weight_values = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]  # 1.0 = classifier only
threshold_values = np.arange(0.40, 0.95, 0.05)

grid_results = []

for w in weight_values:
    for t in threshold_values:
        preds, confs = ensemble_predict_with_threshold(
            clf_probs_5cls, knn_probs_5cls, w, t, CLEAN_CLASS_NAMES, CLASS_NAMES
        )
        mf1 = f1_score(test_labels_6cls, preds, average='macro')
        acc = (preds == test_labels_6cls).mean()
        grid_results.append({
            'clf_weight': w,
            'threshold': round(t, 2),
            'macro_f1': mf1,
            'accuracy': acc,
        })

grid_df = pd.DataFrame(grid_results)
best_row = grid_df.loc[grid_df['macro_f1'].idxmax()]

print(f'Grid search: {len(grid_results)} combinations\n')
print(f'Best configuration:')
print(f'  Classifier weight: {best_row["clf_weight"]:.1f} (k-NN weight: {1-best_row["clf_weight"]:.1f})')
print(f'  Threshold:         {best_row["threshold"]:.2f}')
print(f'  Macro F1:          {best_row["macro_f1"]:.4f}')
print(f'  Accuracy:          {best_row["accuracy"]:.4f}')

In [ ]:
# Heatmap: Macro F1 across weights and thresholds
pivot = grid_df.pivot_table(values='macro_f1', index='clf_weight', columns='threshold')

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGn', ax=ax, cbar_kws={'label': 'Macro F1'})
ax.set_xlabel('Confidence Threshold')
ax.set_ylabel('Classifier Weight (1.0 = classifier only, 0.3 = mostly k-NN)')
ax.set_title('Ensemble Grid Search: Macro F1', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('ensemble_grid_search.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Detailed Evaluation of Best Ensemble

In [ ]:
# Final predictions with best config
best_w = best_row['clf_weight']
best_t = best_row['threshold']

final_preds, final_confs = ensemble_predict_with_threshold(
    clf_probs_5cls, knn_probs_5cls, best_w, best_t, CLEAN_CLASS_NAMES, CLASS_NAMES
)

print(f'Ensemble config: clf_weight={best_w:.1f}, threshold={best_t:.2f}')
print(f'Accuracy:  {(final_preds == test_labels_6cls).mean():.4f}')
print(f'Macro F1:  {f1_score(test_labels_6cls, final_preds, average="macro"):.4f}')
print(f'\n{"="*70}')
print('CLASSIFICATION REPORT (Ensemble)')
print(f'{"="*70}')
print(classification_report(test_labels_6cls, final_preds, target_names=CLASS_NAMES, digits=4))

In [ ]:
cm = confusion_matrix(test_labels_6cls, final_preds)
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title('Confusion Matrix (Recall)', fontweight='bold')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('confusion_ensemble.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Ablation: Compare Individual Models vs Ensemble

In [ ]:
# Classifier only (weight=1.0) at various thresholds
clf_best_f1, clf_best_t = 0, 0
for t in threshold_values:
    preds, _ = ensemble_predict_with_threshold(
        clf_probs_5cls, knn_probs_5cls, 1.0, t, CLEAN_CLASS_NAMES, CLASS_NAMES
    )
    mf1 = f1_score(test_labels_6cls, preds, average='macro')
    if mf1 > clf_best_f1:
        clf_best_f1 = mf1; clf_best_t = t

# k-NN only (weight=0.0) at various thresholds
knn_best_f1, knn_best_t = 0, 0
for t in threshold_values:
    preds, _ = ensemble_predict_with_threshold(
        clf_probs_5cls, knn_probs_5cls, 0.0, t, CLEAN_CLASS_NAMES, CLASS_NAMES
    )
    mf1 = f1_score(test_labels_6cls, preds, average='macro')
    if mf1 > knn_best_f1:
        knn_best_f1 = mf1; knn_best_t = t

ablation = pd.DataFrame({
    'Method': [
        'Classifier only',
        'k-NN only',
        f'Ensemble (w={best_w:.1f})',
    ],
    'Threshold': [clf_best_t, knn_best_t, best_t],
    'Macro F1': [clf_best_f1, knn_best_f1, best_row['macro_f1']],
})

print('=' * 55)
print('ABLATION: Individual vs Ensemble')
print('=' * 55)
print(ablation.to_string(index=False))
print('=' * 55)

improvement = best_row['macro_f1'] - clf_best_f1
print(f'\nEnsemble improvement over classifier alone: {improvement:+.4f} Macro F1')

## 11. Full Project Comparison

In [ ]:
ensemble_f1 = best_row['macro_f1']
ensemble_acc = best_row['accuracy']

full_comparison = pd.DataFrame({
    'Experiment': [
        'EfficientNet-B0 + imbalance mitigation',
        'EfficientNet-B0 baseline',
        'DINOv2 6-class (before relabeling)',
        'DINOv2 + CutMix/MixUp',
        'DINOv2 6-class (after relabeling)',
        'DINOv2 frozen + threshold',
        'DINOv2 unfreeze-1 + threshold',
        f'DINOv2 unfreeze-1 + k-NN ensemble + threshold',
    ],
    'Macro F1': [
        0.7645, 0.8089, 0.8244, 0.7896,
        0.8261, 0.8628, 0.8992,
        ensemble_f1,
    ],
    'Accuracy': [
        0.7667, 0.7833, 0.8167, 0.7833,
        0.8333, 0.8500, 0.9000,
        ensemble_acc,
    ],
    'Test Set': [
        'Original', 'Original', 'Original', 'Original',
        'Relabeled', 'Relabeled', 'Relabeled',
        'Relabeled',
    ]
})

print('=' * 95)
print('FULL PROJECT COMPARISON')
print('=' * 95)
print(full_comparison.to_string(index=False))
print('=' * 95)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))
y = np.arange(len(full_comparison))
w = 0.35

ax.barh(y + w/2, full_comparison['Macro F1'], w, label='Macro F1', color='#4CAF50')
ax.barh(y - w/2, full_comparison['Accuracy'], w, label='Accuracy', color='#2196F3')

ax.set_yticks(y)
ax.set_yticklabels(full_comparison['Experiment'], fontsize=8)
ax.set_xlabel('Score')
ax.set_title('Full Project — Experiment Comparison', fontweight='bold', fontsize=14)
ax.legend(loc='lower right')
ax.set_xlim(0.6, 1.0)
ax.grid(axis='x', alpha=0.3)

for i, (f1v, accv) in enumerate(zip(full_comparison['Macro F1'], full_comparison['Accuracy'])):
    ax.text(f1v + 0.003, i + w/2, f'{f1v:.4f}', va='center', fontsize=7)
    ax.text(accv + 0.003, i - w/2, f'{accv:.4f}', va='center', fontsize=7)

plt.tight_layout()
plt.savefig('full_comparison_with_ensemble.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Summary

### What was done:
1. Trained the best model from previous experiment (DINOv2 unfreeze-1)
2. Extracted fine-tuned features for both train and test sets
3. Found optimal k for k-NN via 5-fold cross-validation
4. Built ensemble by averaging classifier softmax + k-NN probability distributions
5. Grid-searched over ensemble weights (0.3–1.0) and thresholds (0.40–0.90)
6. Compared: classifier only vs k-NN only vs ensemble

### Key design choices:
- **Cosine distance + distance weighting** for k-NN — matches how DINOv2 features are organized in the embedding space
- **L2 normalization** of features before k-NN — ensures cosine distance works correctly
- **Fine-tuned features** (not raw DINOv2) for k-NN — the adapted features from unfreeze-1 are domain-specific
- **Probability averaging** (not vote averaging) — preserves confidence information needed for thresholding
- **Grid search over both weight and threshold** — these two hyperparameters interact, so we optimize jointly